In [9]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

In [10]:
DATA_DIR = Path("../data/raw")

train = pd.read_csv(DATA_DIR / "train.csv")

target = "SalePrice"
id_col = "Id"

y = train[target]
X = train.drop(columns=[target, id_col])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X:", X.shape)
print("y:", y.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X: (1460, 79)
y: (1460,)
X_train: (1168, 79)
X_test: (292, 79)


In [11]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 36
Categorical features: 43


C:\Users\Александр\AppData\Local\Temp\ipykernel_35928\3208774431.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()


In [12]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [13]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return mean_squared_error(np.log1p(y_true), np.log1p(y_pred)) ** 0.5

scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(rmse, greater_is_better=False),
    "r2": make_scorer(r2_score),
    "rmsle": make_scorer(rmsle, greater_is_better=False),
}

In [14]:
base_models = {
    "Ridge": Ridge(),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
}

In [15]:
def evaluate_model_cv(model_name, model, target_strategy):
    base_pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    if target_strategy == "raw":
        estimator = base_pipe
    elif target_strategy == "log1p":
        estimator = TransformedTargetRegressor(
            regressor=base_pipe,
            func=np.log1p,
            inverse_func=np.expm1
        )
    else:
        raise ValueError("target_strategy must be either 'raw' or 'log1p'")
    
    cv_results = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    
    return {
        "model": model_name,
        "target_strategy": target_strategy,
        "MAE_mean": -cv_results["test_mae"].mean(),
        "MAE_std": cv_results["test_mae"].std(),
        "RMSE_mean": -cv_results["test_rmse"].mean(),
        "RMSE_std": cv_results["test_rmse"].std(),
        "R2_mean": cv_results["test_r2"].mean(),
        "R2_std": cv_results["test_r2"].std(),
        "RMSLE_mean": -cv_results["test_rmsle"].mean(),
        "RMSLE_std": cv_results["test_rmsle"].std(),
    }

Блок 8 — raw vs log target comparison

In [16]:
target_results = []

for model_name, model in base_models.items():
    for target_strategy in ["raw", "log1p"]:
        target_results.append(
            evaluate_model_cv(model_name, model, target_strategy)
        )

target_results_df = (
    pd.DataFrame(target_results)
    .sort_values("RMSE_mean")
    .reset_index(drop=True)
)

display(target_results_df)

,model,target_strategy,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,RMSLE_mean,RMSLE_std
0,HistGradientBoostingRegressor,log1p,16860.509585,1530.417781,28548.713699,4280.061428,0.861725,0.027629,0.133938,0.016208
1,GradientBoostingRegressor,raw,16631.255890,1421.396264,28741.968246,3958.720183,0.858184,0.036620,0.134663,0.016847
2,GradientBoostingRegressor,log1p,16282.523015,1691.068921,28943.175592,4471.465702,0.856651,0.036785,0.132726,0.016473
3,HistGradientBoostingRegressor,raw,17279.885742,1823.199437,29404.090922,5353.785125,0.851486,0.045032,0.137359,0.018818
4,RandomForestRegressor,raw,18198.367144,1637.266973,30583.721919,4855.965157,0.838639,0.048042,0.146990,0.020021
5,RandomForestRegressor,log1p,17937.147642,2075.586307,30923.853692,4442.590156,0.836132,0.041777,0.144630,0.020270
6,Ridge,raw,18762.894619,1123.352361,33854.360160,8042.315619,0.802966,0.073065,0.165322,0.012426
7,Ridge,log1p,17651.617970,3044.500372,46589.064849,33368.807210,0.511521,0.612497,0.151241,0.029463


Блок 9 — compare with Stage 4 references

In [17]:
stage4_reference = pd.DataFrame([
    {
        "model": "HistGradientBoostingRegressor",
        "Stage4_MAE": 17279.89,
        "Stage4_RMSE": 29404.09,
        "Stage4_R2": 0.8515,
    },
    {
        "model": "GradientBoostingRegressor",
        "Stage4_MAE": 16686.98,
        "Stage4_RMSE": 29571.13,
        "Stage4_R2": 0.8501,
    },
    {
        "model": "Ridge",
        "Stage4_MAE": 18758.16,
        "Stage4_RMSE": 33857.89,
        "Stage4_R2": 0.8029,
    },
    {
        "model": "RandomForestRegressor",
        "Stage4_MAE": 18198.37,
        "Stage4_RMSE": 30583.72,
        "Stage4_R2": 0.8386,
    },
])

comparison_with_stage4 = target_results_df.merge(
    stage4_reference,
    on="model",
    how="left"
)

comparison_with_stage4["RMSE_diff_vs_Stage4"] = (
    comparison_with_stage4["RMSE_mean"] - comparison_with_stage4["Stage4_RMSE"]
)

display(comparison_with_stage4.sort_values("RMSE_mean"))

,model,target_strategy,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,RMSLE_mean,RMSLE_std,Stage4_MAE,Stage4_RMSE,Stage4_R2,RMSE_diff_vs_Stage4
0,HistGradientBoostingRegressor,log1p,16860.509585,1530.417781,28548.713699,4280.061428,0.861725,0.027629,0.133938,0.016208,17279.89,29404.09,0.8515,-855.376301
1,GradientBoostingRegressor,raw,16631.255890,1421.396264,28741.968246,3958.720183,0.858184,0.036620,0.134663,0.016847,16686.98,29571.13,0.8501,-829.161754
2,GradientBoostingRegressor,log1p,16282.523015,1691.068921,28943.175592,4471.465702,0.856651,0.036785,0.132726,0.016473,16686.98,29571.13,0.8501,-627.954408
3,HistGradientBoostingRegressor,raw,17279.885742,1823.199437,29404.090922,5353.785125,0.851486,0.045032,0.137359,0.018818,17279.89,29404.09,0.8515,0.000922
4,RandomForestRegressor,raw,18198.367144,1637.266973,30583.721919,4855.965157,0.838639,0.048042,0.146990,0.020021,18198.37,30583.72,0.8386,0.001919
5,RandomForestRegressor,log1p,17937.147642,2075.586307,30923.853692,4442.590156,0.836132,0.041777,0.144630,0.020270,18198.37,30583.72,0.8386,340.133692
6,Ridge,raw,18762.894619,1123.352361,33854.360160,8042.315619,0.802966,0.073065,0.165322,0.012426,18758.16,33857.89,0.8029,-3.529840
7,Ridge,log1p,17651.617970,3044.500372,46589.064849,33368.807210,0.511521,0.612497,0.151241,0.029463,18758.16,33857.89,0.8029,12731.174849


Блок 10 — select best candidates for limited tuning

HistGradientBoostingRegressor	log1p
GradientBoostingRegressor	raw

Блок 11 — tuning search spaces

In [ ]:
hgb_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingRegressor(random_state=42))
])

hgb_log_estimator = TransformedTargetRegressor(
    regressor=hgb_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

hgb_param_grid = {
    "regressor__model__learning_rate": [0.05, 0.1],
    "regressor__model__max_leaf_nodes": [15, 31],
    "regressor__model__l2_regularization": [0.0, 0.1],
}


GradientBoostingRegressor RAW

In [30]:
gbr_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])



gbr_param_grid = {
    "regressor__model__n_estimators": [100, 200],
    "regressor__model__learning_rate": [0.05, 0.1],
    "regressor__model__max_depth": [2, 3],
}

Блок 12 — GridSearchCV helper

In [31]:
scoring = {
    "neg_mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "neg_rmse": make_scorer(rmse, greater_is_better=False),
    "r2": make_scorer(r2_score),
    "neg_rmsle": make_scorer(rmsle, greater_is_better=False),
}

def run_grid_search(name, estimator, param_grid):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scoring,
        refit="neg_rmse",
        cv=5,
        n_jobs=-1,
        return_train_score=False
    )
    
    grid.fit(X_train, y_train)
    
    best_idx = grid.best_index_
    
    result = {
        "model": name,
        "best_params": grid.best_params_,
        "MAE_mean": -grid.cv_results_["mean_test_neg_mae"][best_idx],
        "MAE_std": grid.cv_results_["std_test_neg_mae"][best_idx],
        "RMSE_mean": -grid.cv_results_["mean_test_neg_rmse"][best_idx],
        "RMSE_std": grid.cv_results_["std_test_neg_rmse"][best_idx],
        "R2_mean": grid.cv_results_["mean_test_r2"][best_idx],
        "R2_std": grid.cv_results_["std_test_r2"][best_idx],
        "RMSLE_mean": -grid.cv_results_["mean_test_neg_rmsle"][best_idx],
        "RMSLE_std": grid.cv_results_["std_test_neg_rmsle"][best_idx],
    }
    
    return grid, result

Блок 13 — run limited tuning

In [32]:
hgb_grid, hgb_tuned_result = run_grid_search(
    name="HistGradientBoostingRegressor_log1p_tuned",
    estimator=hgb_log_estimator,
    param_grid=hgb_param_grid
)

gbr_grid, gbr_tuned_result = run_grid_search(
    name="GradientBoostingRegressor_log1p_tuned",
    estimator=gbr_log_estimator,
    param_grid=gbr_param_grid
)

tuned_results_df = pd.DataFrame([
    hgb_tuned_result,
    gbr_tuned_result,
]).sort_values("RMSE_mean")

display(tuned_results_df)

,model,best_params,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,RMSLE_mean,RMSLE_std
0,HistGradientBoostingRegressor_log1p_tuned,"{'regressor__model__l2_regularization': 0.1, '...",16484.785143,1495.834044,28110.148516,3764.046555,0.866307,0.021476,0.132635,0.014089
1,GradientBoostingRegressor_log1p_tuned,"{'regressor__model__learning_rate': 0.1, 'regr...",15898.705345,1488.531775,28271.608505,3949.819804,0.863365,0.032167,0.130310,0.015486


Блок 14 — tuned vs untuned comparison

In [33]:
best_untuned = (
    target_results_df
    .sort_values("RMSE_mean")
    .head(8)
)

display(best_untuned)
display(tuned_results_df)

,model,target_strategy,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,RMSLE_mean,RMSLE_std
0,HistGradientBoostingRegressor,log1p,16860.509585,1530.417781,28548.713699,4280.061428,0.861725,0.027629,0.133938,0.016208
1,GradientBoostingRegressor,raw,16631.255890,1421.396264,28741.968246,3958.720183,0.858184,0.036620,0.134663,0.016847
2,GradientBoostingRegressor,log1p,16282.523015,1691.068921,28943.175592,4471.465702,0.856651,0.036785,0.132726,0.016473
3,HistGradientBoostingRegressor,raw,17279.885742,1823.199437,29404.090922,5353.785125,0.851486,0.045032,0.137359,0.018818
4,RandomForestRegressor,raw,18198.367144,1637.266973,30583.721919,4855.965157,0.838639,0.048042,0.146990,0.020021
5,RandomForestRegressor,log1p,17937.147642,2075.586307,30923.853692,4442.590156,0.836132,0.041777,0.144630,0.020270
6,Ridge,raw,18762.894619,1123.352361,33854.360160,8042.315619,0.802966,0.073065,0.165322,0.012426
7,Ridge,log1p,17651.617970,3044.500372,46589.064849,33368.807210,0.511521,0.612497,0.151241,0.029463


,model,best_params,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,RMSLE_mean,RMSLE_std
0,HistGradientBoostingRegressor_log1p_tuned,"{'regressor__model__l2_regularization': 0.1, '...",16484.785143,1495.834044,28110.148516,3764.046555,0.866307,0.021476,0.132635,0.014089
1,GradientBoostingRegressor_log1p_tuned,"{'regressor__model__learning_rate': 0.1, 'regr...",15898.705345,1488.531775,28271.608505,3949.819804,0.863365,0.032167,0.130310,0.015486


# Stage 5 conclusions

### Target strategy
- Raw SalePrice and log1p(SalePrice) were compared using CV on X_train only.
- log1p was implemented using TransformedTargetRegressor.
- Metrics were reported on the original dollar scale after inverse transformation.

### Limited tuning
- Only the best 1–2 candidate families were tuned.
- Search spaces were intentionally small.
- No local test evaluation was performed.
- No official Kaggle test.csv was used.


Best RMSE candidate:
- HistGradientBoostingRegressor with log1p target and limited tuning.

Best MAE candidate:
- GradientBoostingRegressor with log1p target and limited tuning.

Compared with Stage 4:
- Stage 4 best RMSE was HistGradientBoostingRegressor raw target: RMSE ≈ 29,404.
- Stage 5 best RMSE is tuned HistGradientBoostingRegressor with log1p target: RMSE ≈ 28,110.
- Improvement ≈ 1,294 RMSE.

Stage 5 selected model for later final holdout evaluation:
- HistGradientBoostingRegressor_log1p_tuned by RMSE.
- GradientBoostingRegressor_log1p_tuned remains an alternative if MAE is prioritized.